## 0. Prérequis : Créer une DB et la table dans postgres

CREATE TABLE reviews (
    id BIGINT PRIMARY KEY,
    listing_id BIGINT,
    date DATE,
    reviewer_id BIGINT,
    reviewer_name TEXT,
    comments TEXT
);


# 1. Création de la DB à partir du fichier reviews.csv

In [7]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import BigInteger, Text, Date

user = "noam"
password = "noam"
host = "localhost"
port = "5432"
database = "projet_big_data"

# Connexion via SQLAlchemy
engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{database}")

df = pd.read_csv("../data/source/reviews.csv", parse_dates=["date"])

df.drop_duplicates(subset=["id"], inplace=True)

df.to_sql("reviews", engine, index=False, if_exists="replace", dtype={
    "id": BigInteger(),
    "listing_id": BigInteger(),
    "reviewer_id": BigInteger(),
    "reviewer_name": Text(),
    "comments": Text(),
    "date": Date()
})



849

## 2. Postgres to Bronze


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("extract reviews from postgres") \
    .config("spark.jars", "/Users/noam/Downloads/postgresql-42.7.4.jar") \
    .enableHiveSupport() \
    .getOrCreate()

df_reviews = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/projet_big_data") \
    .option("dbtable", "reviews") \
    .option("user", "noam") \
    .option("password", "noam") \
    .option("driver", "org.postgresql.Driver") \
    .load()

df_reviews.show(5)


25/05/02 11:07:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


+----------+--------+----------+-----------+-------------+--------------------+
|listing_id|      id|      date|reviewer_id|reviewer_name|            comments|
+----------+--------+----------+-----------+-------------+--------------------+
|   7202016|38917982|2015-07-19|   28943674|       Bianca|Cute and cozy pla...|
|   7202016|39087409|2015-07-20|   32440555|        Frank|Kelly has a great...|
|   7202016|39820030|2015-07-26|   37722850|          Ian|Very spacious apa...|
|   7202016|40813543|2015-08-02|   33671805|       George|Close to Seattle ...|
|   7202016|41986501|2015-08-10|   34959538|         Ming|Kelly was a great...|
+----------+--------+----------+-----------+-------------+--------------------+
only showing top 5 rows



### Ajout de la colonne processing date et création de colonne pour le partitionnement

In [5]:
from pyspark.sql.functions import current_date, year, month, dayofmonth

df_reviews = df_reviews.withColumn("processing_date", current_date())
df_reviews = df_reviews \
    .withColumn("year", year("processing_date")) \
    .withColumn("month", month("processing_date")) \
    .withColumn("day", dayofmonth("processing_date"))


### Ecriture en parquet avec partitionnement

In [8]:
df_reviews.write \
    .partitionBy("year", "month", "day") \
    .mode("overwrite") \
    .parquet("../data/bronze/reviews/")


In [27]:
spark.stop()

## 3. Listing

### Division en plusieurs lots pour batch

In [10]:
import os
import pandas as pd

input_file = "../data/source/listings.csv"
batch_dir = "../data/source2/batch_input_listings"
batch_size = 100  # lignes par lot

os.makedirs(batch_dir, exist_ok=True)

df = pd.read_csv(input_file)

for i in range(0, len(df), batch_size):
    batch = df.iloc[i:i + batch_size]
    batch_file = os.path.join(batch_dir, f"batch_{i // batch_size + 1}.csv")
    batch.to_csv(batch_file, index=False)

print(f"✅ {len(df)} lignes découpées en lots de {batch_size} dans {batch_dir}")


✅ 3818 lignes découpées en lots de 100 dans ../data/source2/batch_input_listings


### Ecriture en parquet et partitionnement par date 

In [12]:
import os
import time
from pyspark.sql.functions import current_date, year, month, dayofmonth

batch_dir = "../data/source2/batch_input_listings"
bronze_dir = "../data/bronze/listings"

os.makedirs(bronze_dir, exist_ok=True)

batch_files = sorted([f for f in os.listdir(batch_dir) if f.endswith(".csv")])

for batch_file in batch_files:
    path = os.path.join(batch_dir, batch_file)
    print(f"🔄 Traitement du fichier : {batch_file}")

    df_batch = spark.read.format("csv").option("header", True).load(path)

    #date traitement pour partitionnement
    df_batch = df_batch.withColumn("processing_date", current_date())
    df_batch = df_batch.withColumn("year", year("processing_date")) \
                       .withColumn("month", month("processing_date")) \
                       .withColumn("day", dayofmonth("processing_date"))

    # Répartition des données en 3 partitions pour la parallélisation
    df_batch = df_batch.repartition(3)

    #partitionnement temporel
    df_batch.write \
        .mode("append") \
        .partitionBy("year", "month", "day") \
        .parquet(bronze_dir)

    time.sleep(1) 


🔄 Traitement du fichier : batch_1.csv


25/05/02 11:07:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


🔄 Traitement du fichier : batch_10.csv


🔄 Traitement du fichier : batch_11.csv


🔄 Traitement du fichier : batch_12.csv


🔄 Traitement du fichier : batch_13.csv
🔄 Traitement du fichier : batch_14.csv
🔄 Traitement du fichier : batch_15.csv
🔄 Traitement du fichier : batch_16.csv
🔄 Traitement du fichier : batch_17.csv
🔄 Traitement du fichier : batch_18.csv
🔄 Traitement du fichier : batch_19.csv
🔄 Traitement du fichier : batch_2.csv
🔄 Traitement du fichier : batch_20.csv
🔄 Traitement du fichier : batch_21.csv
🔄 Traitement du fichier : batch_22.csv
🔄 Traitement du fichier : batch_23.csv
🔄 Traitement du fichier : batch_24.csv
🔄 Traitement du fichier : batch_25.csv
🔄 Traitement du fichier : batch_26.csv
🔄 Traitement du fichier : batch_27.csv
🔄 Traitement du fichier : batch_28.csv
🔄 Traitement du fichier : batch_29.csv
🔄 Traitement du fichier : batch_3.csv
🔄 Traitement du fichier : batch_30.csv
🔄 Traitement du fichier : batch_31.csv
🔄 Traitement du fichier : batch_32.csv
🔄 Traitement du fichier : batch_33.csv
🔄 Traitement du fichier : batch_34.csv
🔄 Traitement du fichier : batch_35.csv
🔄 Traitement du fichier : b

🔄 Traitement du fichier : batch_39.csv
🔄 Traitement du fichier : batch_4.csv
🔄 Traitement du fichier : batch_5.csv
🔄 Traitement du fichier : batch_6.csv
🔄 Traitement du fichier : batch_7.csv
🔄 Traitement du fichier : batch_8.csv
🔄 Traitement du fichier : batch_9.csv


# 4. Reviews en Silver

In [53]:
from pyspark.sql.functions import year, lower, length, col

df_reviews = spark.read.parquet("../data/bronze/reviews")

df_reviews_clean = df_reviews \
    .withColumn("year_review", year("date")) \
    .withColumn("comments", lower(col("comments"))) \
    .filter(length("comments") > 10)

df_reviews_clean.write.mode("overwrite").parquet("../data/silver/reviews")

df_reviews_clean.write.mode("overwrite").saveAsTable("silver_reviews")

print("✅ Table Hive 'silver_reviews' créée.")


25/05/02 02:24:28 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
25/05/02 02:24:28 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
25/05/02 02:24:33 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
25/05/02 02:24:33 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore noam@192.168.1.129
25/05/02 02:24:33 WARN ObjectStore: Failed to get database default, returning NoSuchObjectException
25/05/02 02:24:37 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
25/05/02 02:24:37 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
25/05/02 02:24:37 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
25/05/02 02:24:37 WARN 

✅ Table Hive 'silver_reviews' créée.


25/05/02 02:24:38 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


# 5. Listings en Silver

In [57]:
from pyspark.sql.functions import regexp_replace, col

df_listings = spark.read.parquet("../data/bronze/listings")

colonnes_utiles = [
    "id", "host_id",
    "property_type", "room_type", "accommodates", "bedrooms", "beds",
    "neighbourhood_cleansed", "latitude", "longitude",
    "host_response_time", "host_response_rate", "host_is_superhost",
    "price", "cleaning_fee", "security_deposit", "extra_people",
    "review_scores_rating", "reviews_per_month",
    "first_review", "last_review"
]
df_listings = df_listings.select(*colonnes_utiles)

df_listings_clean = df_listings.withColumn("price", regexp_replace("price", "[$,]", "").cast("float")) \
    .withColumn("cleaning_fee", regexp_replace("cleaning_fee", "[$,]", "").cast("float")) \
    .withColumn("security_deposit", regexp_replace("security_deposit", "[$,]", "").cast("float")) \
    .withColumn("extra_people", regexp_replace("extra_people", "[$,]", "").cast("float")) \
    .withColumn("host_response_rate", regexp_replace("host_response_rate", "[% ]", "").cast("float")) \
    .withColumn("review_scores_rating", col("review_scores_rating").cast("float")) \
    .withColumn("reviews_per_month", col("reviews_per_month").cast("float")) \
    .withColumn("accommodates", col("accommodates").cast("int")) \
    .withColumn("bedrooms", col("bedrooms").cast("float")) \
    .withColumn("beds", col("beds").cast("float"))

df_listings_clean = df_listings_clean.filter(col("price").isNotNull()) \
    .filter(col("room_type").isNotNull()) \
    .filter(col("host_id").isNotNull()) \
    .filter(col("neighbourhood_cleansed").isNotNull())

# 5. Sauvegarde en Parquet
df_listings_clean.write.mode("overwrite").parquet("../data/silver/listings")

# 6. Sauvegarde dans Hive
df_listings_clean.write.mode("overwrite").saveAsTable("silver_listings")

print("✅ Table Hive 'silver_listings' créée avec succès.")


25/05/02 02:25:29 WARN MemoryManager: Total allocation exceeds 95,00% (906 992 014 bytes) of heap memory
Scaling row group sizes to 96,54% for 7 writers
25/05/02 02:25:29 WARN MemoryManager: Total allocation exceeds 95,00% (906 992 014 bytes) of heap memory
Scaling row group sizes to 84,47% for 8 writers
25/05/02 02:25:31 WARN MemoryManager: Total allocation exceeds 95,00% (906 992 014 bytes) of heap memory
Scaling row group sizes to 96,54% for 7 writers
25/05/02 02:25:32 WARN MemoryManager: Total allocation exceeds 95,00% (906 992 014 bytes) of heap memory
Scaling row group sizes to 96,54% for 7 writers
25/05/02 02:25:32 WARN MemoryManager: Total allocation exceeds 95,00% (906 992 014 bytes) of heap memory
Scaling row group sizes to 84,47% for 8 writers
25/05/02 02:25:33 WARN MemoryManager: Total allocation exceeds 95,00% (906 992 014 bytes) of heap memory
Scaling row group sizes to 96,54% for 7 writers


✅ Table Hive 'silver_listings' créée.


In [61]:
spark.sql("SELECT * FROM silver_reviews LIMIT 5").show(5)
spark.sql("SELECT * FROM silver_listings LIMIT 5").show(1)


+----------+--------+----------+-----------+-------------+--------------------+---------------+----+-----+---+-----------+
|listing_id|      id|      date|reviewer_id|reviewer_name|            comments|processing_date|year|month|day|year_review|
+----------+--------+----------+-----------+-------------+--------------------+---------------+----+-----+---+-----------+
|   7202016|38917982|2015-07-19|   28943674|       Bianca|cute and cozy pla...|     2025-05-02|2025|    5|  2|       2015|
|   7202016|39087409|2015-07-20|   32440555|        Frank|kelly has a great...|     2025-05-02|2025|    5|  2|       2015|
|   7202016|39820030|2015-07-26|   37722850|          Ian|very spacious apa...|     2025-05-02|2025|    5|  2|       2015|
|   7202016|40813543|2015-08-02|   33671805|       George|close to seattle ...|     2025-05-02|2025|    5|  2|       2015|
|   7202016|41986501|2015-08-10|   34959538|         Ming|kelly was a great...|     2025-05-02|2025|    5|  2|       2015|
+----------+----

# GOLD jointure entre Reviews et Listings

In [ ]:
df_reviews = spark.table("silver_reviews")
df_listings = spark.table("silver_listings")


In [ ]:
df_enriched = df_reviews.join(
    df_listings,
    df_reviews["listing_id"] == df_listings["id"],
    how="inner" 
)


In [ ]:
df_enriched.write.mode("overwrite").saveAsTable("gold_reviews_enriched")
